In [1]:
import requests
import pandas as pd
from pathlib import Path

In [2]:
ORTHANC = "https://orthanc.unboxed-2026.ovh"
AUTH    = ("unboxed", "unboxed2026")

print("─" * 50)
try:
    r = requests.get(f"{ORTHANC}/system", auth=AUTH, timeout=5)
    if r.status_code == 200:
        info = r.json()
        print(f"  ✅ Orthanc connecté")
        print(f"  Version     : {info['Version']}")
        print(f"  AET         : {info['DicomAet']}")
        print(f"  Storage     : {info.get('StorageSize', 'N/A')}")
    else:
        print(f"  ❌ Erreur HTTP {r.status_code}")
except requests.exceptions.ConnectionError:
    print("  ❌ Impossible de joindre Orthanc (réseau privé requis)")
except requests.exceptions.Timeout:
    print("  ❌ Timeout — serveur Orthanc non joignable")
print("─" * 50)

──────────────────────────────────────────────────
  ✅ Orthanc connecté
  Version     : mainline
  AET         : UNBOXED
  Storage     : N/A
──────────────────────────────────────────────────


In [1]:
studies_ids = requests.get(f"{ORTHANC}/studies", auth=AUTH, timeout=5).json()
print(f"  📊 {len(studies_ids)} étude(s) DICOM dans Orthanc\n")

if not studies_ids:
    print("  ℹ️  Le dataset sera chargé avant le hackathon.")
else:
    rows = []
    for sid in studies_ids:
        info = requests.get(f"{ORTHANC}/studies/{sid}", auth=AUTH).json()
        t = info.get("MainDicomTags", {})
        print(t)
        rows.append({
            "ID"          : sid[:12] + "…",
            "Patient"     : t.get("PatientID", "-"),
            "Description" : t.get("StudyDescription", "-"),
            "Date"        : t.get("StudyDate", "-"),
            "Modalité"    : t.get("ModalitiesInStudy", "-"),
        })
    df = pd.DataFrame(rows)
    display(df)

NameError: name 'requests' is not defined

In [4]:
def download_study(study_id: str, out_dir: str = "/home/jovyan/work") -> str:
    """Télécharge une étude complète (.zip) depuis Orthanc."""
    dest = Path(out_dir) / f"study_{study_id[:8]}.zip"
    print(f"  ⬇️  Téléchargement de l'étude {study_id[:12]}…")
    with requests.get(f"{ORTHANC}/studies/{study_id}/archive",
                      auth=AUTH, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = 0
        with open(dest, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
                total += len(chunk)
    print(f"  ✅ Sauvegardé : {dest}  ({total/1e6:.1f} Mo)")
    !unzip {dest} -A
    return str(dest)

# ── Utilisation ──────────────────────────────────────────────────────
# study_id = "1f8a3b2c-..."   # ← obtenu depuis la cellule ⑤
# download_study(study_id)
for sid in df['ID']:
    print(sid)
    download_study(sid)
print("  Fonction download_study() prête ✓")

1fc1a205-400…
  ⬇️  Téléchargement de l'étude 1fc1a205-400…


HTTPError: 404 Client Error: Not Found for url: https://orthanc.unboxed-2026.ovh/studies/1fc1a205-400%E2%80%A6/archive